# Silver Layer — Cleaned & Enriched Data
**Medallion Healthcare Analytics Platform**  
Celebal Excellence Internship 2025

> The Silver layer applies data quality rules to Bronze data: deduplication, outlier removal, null handling, standardisation, and patient registry enrichment. Output is the single source of truth for ML and Gold layers.

## Silver Layer Transformation Pipeline
```
BRONZE (raw, 417,866 rows)
    ↓ 1. Deduplication
    ↓ 2. Outlier removal (impossible vitals)
    ↓ 3. Null handling (forward-fill per patient)
    ↓ 4. Type casting & normalisation
    ↓ 5. Registry join (add ward, condition, demographics)
    ↓ 6. Derived features (EWS score, risk band, vital_flag)
SILVER (clean, ~3,131 rows)
```

## 1. Setup

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath('.'))))
sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
from config.settings import BRONZE_VITALS_PATH, SILVER_VITALS_PATH, VITAL_THRESHOLDS

## 2. Load Bronze Data

In [ ]:
bronze = pd.read_parquet(BRONZE_VITALS_PATH)
print(f'Bronze rows: {len(bronze):,}')
print(f'Columns: {len(bronze.columns)}')
bronze[['patient_id','heart_rate','spo2_pct','temperature_c','systolic_bp']].describe().round(2)

## 3. Step 1 — Deduplication
Remove exact `(patient_id, timestamp)` duplicates — keep last record (most recent sensor reading).

In [ ]:
before = len(bronze)
bronze_dedup = bronze.drop_duplicates(subset=['patient_id','timestamp'], keep='last')
print(f'Before: {before:,} rows')
print(f'After:  {len(bronze_dedup):,} rows')
print(f'Removed: {before - len(bronze_dedup):,} duplicates ({(before-len(bronze_dedup))/before*100:.1f}%)')

## 4. Step 2 — Outlier Removal
Drop physiologically impossible readings (e.g. HR=0, SpO₂=150%).

In [ ]:
print('Clinical thresholds applied:')
for col, bounds in VITAL_THRESHOLDS.items():
    print(f'  {col:<20} [{bounds["low"]} – {bounds["high"]}]')

mask = pd.Series(True, index=bronze_dedup.index)
for col, bounds in VITAL_THRESHOLDS.items():
    if col in bronze_dedup.columns:
        mask &= (bronze_dedup[col] >= bounds['low']) & (bronze_dedup[col] <= bounds['high'])

before2 = len(bronze_dedup)
silver_clean = bronze_dedup[mask].copy()
print(f'\nRows removed: {before2 - len(silver_clean):,}')
print(f'Silver rows:  {len(silver_clean):,}')

## 5. Step 3 — Null Handling
Forward-fill numeric nulls within each patient (carry last valid reading), then drop any remaining.

In [ ]:
numeric_cols = silver_clean.select_dtypes(include=[np.number]).columns.tolist()
null_before = silver_clean[numeric_cols].isnull().sum().sum()
print(f'Nulls before: {null_before}')
silver_clean[numeric_cols] = (
    silver_clean.sort_values(['patient_id','timestamp'])
    .groupby('patient_id')[numeric_cols]
    .ffill()
)
null_after = silver_clean[numeric_cols].isnull().sum().sum()
print(f'Nulls after forward-fill: {null_after}')

## 6. Step 4 — Derived Features
Compute EWS (Early Warning Score), vital_flag, and risk_band — all used by Gold layer and ML model.

In [ ]:
silver_clean['vital_flag'] = (
    (silver_clean['heart_rate']       > 100) | (silver_clean['heart_rate']       < 60) |
    (silver_clean['spo2_pct']         < 94) |
    (silver_clean['respiratory_rate'] > 20)  | (silver_clean['respiratory_rate'] < 12) |
    (silver_clean['systolic_bp']      < 100) | (silver_clean['systolic_bp']       > 160) |
    (silver_clean['temperature_c']    > 38.0)| (silver_clean['temperature_c']    < 36.0)
).astype(int)

silver_clean['ews_score'] = (
    (silver_clean['respiratory_rate'] > 24).astype(int) * 3 +
    (silver_clean['spo2_pct']         < 92).astype(int) * 3 +
    (silver_clean['systolic_bp']      < 90).astype(int) * 3 +
    (silver_clean['heart_rate']       > 130).astype(int) * 3 +
    (silver_clean['temperature_c']    > 39.0).astype(int) * 2 +
    (silver_clean['sepsis_risk_score']> 0.6).astype(int) * 2 +
    (silver_clean['nurse_alert']      == 1).astype(int) * 2
)

silver_clean['risk_band'] = pd.cut(
    silver_clean['ews_score'],
    bins=[-1, 0, 3, 6, 100],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print('Risk band distribution:')
print(silver_clean['risk_band'].value_counts())
print(f'\nAbnormal vital readings: {silver_clean["vital_flag"].sum():,} ({silver_clean["vital_flag"].mean()*100:.1f}%)')

## 7. Run via Pipeline Class

In [ ]:
from pipeline.silver_layer import SilverLayer
sl = SilverLayer()
df_silver = sl.run()
print(f'\nFinal Silver rows: {len(df_silver):,}')
print(f'Columns: {len(df_silver.columns)}')
df_silver.head(3)

## 8. Quality Report

In [ ]:
print('=== Silver Layer Quality Report ===')
print(f'Input (Bronze):  {len(bronze):>10,} rows')
print(f'Output (Silver): {len(df_silver):>10,} rows')
print(f'Reduction:       {(1-len(df_silver)/len(bronze))*100:.1f}%')
print(f'Patients:        {df_silver["patient_id"].nunique()}')
print(f'High/Critical:   {(df_silver["risk_band"].isin(["High","Critical"])).sum()}')
print(f'Nulls remaining: {df_silver.isnull().sum().sum()}')
print(f'Duplicates:      {df_silver.duplicated(["patient_id","timestamp"]).sum()}')

## Databricks / PySpark Equivalent
```python
# PySpark Silver transformation
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read Bronze Delta table
bronze_df = spark.read.format('delta').load('/mnt/bronze/vitals')

# Deduplication
window = Window.partitionBy('patient_id').orderBy(F.desc('timestamp'))
silver_df = (bronze_df
    .withColumn('row_num', F.row_number().over(window))
    .filter('row_num = 1')
    .drop('row_num'))

# Outlier removal
silver_df = silver_df.filter(
    (F.col('heart_rate').between(40, 180)) &
    (F.col('spo2_pct').between(70, 100))
)

# Write Silver
silver_df.write.format('delta').mode('overwrite').save('/mnt/silver/vitals')
```